# 4.8 Gaussian Mixture Models & EM

Gaussian mixtures treat cluster ownership as probabilities and alternate between responsibility and parameter updates.

The E-step normalizes competing explanations; the M-step uses weighted averages to move means, covariances, and mixture weights. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(7)


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def scaled(X):
    return StandardScaler().fit_transform(X)


def plot_xy(X):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=0).fit_transform(X)


def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    if len(np.unique(labels)) >= len(labels):
        return float("nan")
    return float(silhouette_score(X, labels))

def gaussian_pdf(X, mean, cov):
    dim = X.shape[1]
    centered = X - mean
    inv = np.linalg.pinv(cov)
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        logdet = np.log(np.maximum(np.linalg.det(cov + 1e-6 * np.eye(dim)), 1e-12))
    quad = np.sum((centered @ inv) * centered, axis=1)
    log_norm = -0.5 * (dim * np.log(2.0 * np.pi) + logdet)
    return np.exp(log_norm - 0.5 * quad)


def gmm_em(X, n_components, max_iter=60, reg=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    n, dim = X.shape
    idx = rng.choice(n, size=n_components, replace=False)
    means = X[idx].copy()
    covs = np.array([np.cov(X.T) + reg * np.eye(dim) for _ in range(n_components)])
    weights = np.ones(n_components) / n_components
    loglikes = []

    for step in range(max_iter):
        q = np.zeros((n, n_components))
        for k in range(n_components):
            q[:, k] = weights[k] * gaussian_pdf(X, means[k], covs[k])
        normalizer = np.maximum(q.sum(axis=1, keepdims=True), 1e-300)
        resp = q / normalizer
        loglikes.append(float(np.sum(np.log(normalizer))))
        Nk = np.maximum(resp.sum(axis=0), 1e-12)
        weights = Nk / n
        means = (resp.T @ X) / Nk[:, None]
        for k in range(n_components):
            centered = X - means[k]
            covs[k] = (resp[:, k][:, None] * centered).T @ centered
            covs[k] = covs[k] / Nk[k]
            covs[k] = covs[k] + reg * np.eye(dim)
        if len(loglikes) > 2 and abs(loglikes[-1] - loglikes[-2]) < 1e-4:
            break

    return resp, means, covs, weights, loglikes


## The concept, built once on D1

The lesson's EM core is the same normalization and weighted update:

$$r_{ik}=\frac{q_{ik}}{\sum_j q_{ij}},\qquad \mu_k=\frac{\sum_i r_{ik}x_i}{\sum_i r_{ik}}$$

We assert the exact toy responsibilities and weighted mean before fitting a full-covariance mixture.

In [ ]:

q = np.array([0.30, 0.70])
r = q / q.sum()
assert np.allclose(r, [0.30, 0.70])

x = np.array([0.0, 1.0, 4.0])
w = np.array([0.8, 0.6, 0.1])
mu = np.sum(w * x) / np.sum(w)
assert round(float(mu), 3) == 0.667

print("responsibility audit", r)
print("M-step mean audit", round(float(mu), 3))


The reusable method below implements real EM with full covariance matrices, regularization, and a log-likelihood trace. The notebook uses silhouette for the shared F2 comparison and keeps likelihood as a diagnostic.

In [ ]:

X_d1 = scaled(cluster_ladder()[0][1])
resp_d1, means_d1, covs_d1, mix_d1, ll_d1 = gmm_em(X_d1, n_components=3, seed=0)
labels_d1 = np.argmax(resp_d1, axis=1)

print("D1 responsibilities")
print(np.round(resp_d1, 3))
print("D1 mixture weights", np.round(mix_d1, 3))
print("D1 final loglike", round(ll_d1[-1], 3))
print("D1 labels", labels_d1.tolist())


## The dataset ladder

We use the shared F2 clustering ladder exactly once in the setup cell: hand points, clean blobs, anisotropic overlap, real Iris, and real digits 0–3. The hidden labels are kept only for audit metrics such as ARI; the clustering methods never train on them.

In [ ]:

rungs = cluster_ladder()

for idx, (name, X, y_true, k) in enumerate(rungs, start=1):
    print(idx, name, "shape=", X.shape, "k=", k, "labels=", np.unique(y_true).tolist())
    print("sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same method across D1–D5

Each rung is scaled, fit with the same EM implementation, converted to labels by responsibility argmax, and scored by silhouette.

In [ ]:

results = []
artifacts = []

for idx, (name, X, y_true, k) in enumerate(cluster_ladder(), start=1):
    Xs = scaled(X)
    resp, means, covs, mix, loglikes = gmm_em(Xs, n_components=k, reg=1e-4, seed=idx)
    labels = np.argmax(resp, axis=1)
    score = safe_silhouette(Xs, labels)
    ari = adjusted_rand_score(y_true, labels)
    min_mix = float(np.min(mix))
    results.append({"rung": idx, "name": name, "score": score, "ari": ari, "min_mix": min_mix, "loglike": loglikes[-1]})
    artifacts.append((Xs, labels, means, covs, resp))

for row in results:
    print(row["rung"], row["name"], "silhouette=", round(row["score"], 3), "ARI=", round(row["ari"], 3), "min_mix=", round(row["min_mix"], 3))


## Results visualization

The panel view shows responsibility-argmax clusters and approximate Gaussian ellipses for two-dimensional projected coordinates; the curve shows silhouette versus rung.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, row, artifact in zip(axes, results, artifacts):
    Xs, labels, means, covs, resp = artifact
    coords = plot_xy(Xs)
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=18, alpha=0.75)
    if Xs.shape[1] == 2:
        for mean in means:
            ax.scatter(mean[0], mean[1], c="black", marker="x", s=50)
    ax.set_title(f"D{row['rung']} sil={row['score']:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

plt.figure(figsize=(6, 3))
plt.plot([r["rung"] for r in results], [r["score"] for r in results], marker="o")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("ladder rung")
plt.ylabel("silhouette")
plt.title("GMM-EM silhouette vs complexity")
plt.show()


## Pitfall on D5: covariance and initialization collapse

Full-covariance EM can create tiny components or unstable covariances on high-dimensional digits. We reproduce the risk with weak regularization, then fix it with stronger regularization and multiple seeds.

In [ ]:

name, X5, y5, k5 = cluster_ladder()[-1]
Xs5 = scaled(X5)

resp_bad, means_bad, covs_bad, mix_bad, ll_bad = gmm_em(Xs5, n_components=k5, reg=1e-10, seed=2)
labels_bad = np.argmax(resp_bad, axis=1)
bad_score = safe_silhouette(Xs5, labels_bad)
bad_min_mix = float(np.min(mix_bad))

best = None

for seed in [0, 1, 2, 3, 4]:
    resp_fix, means_fix, covs_fix, mix_fix, ll_fix = gmm_em(Xs5, n_components=k5, reg=1e-3, seed=seed)
    labels_fix = np.argmax(resp_fix, axis=1)
    score_fix = safe_silhouette(Xs5, labels_fix)
    candidate = (score_fix, float(np.min(mix_fix)), seed)
    if best is None or candidate[0] > best[0]:
        best = candidate

print("weak regularization", round(bad_score, 3), "min_mix=", round(bad_min_mix, 4))
print("regularized multi-seed", round(best[0], 3), "min_mix=", round(best[1], 4), "seed=", best[2])


## Evaluate it + Practice

- Metric: track silhouette from responsibility argmax on every rung and compare against a no-skill baseline such as random labels with the same number of clusters.
- Sanity check: D1 and D2 should be visibly easier than D5; if not, inspect scaling and assignments before trusting the curve.
- Ablation: set covariance regularization near zero or run one seed only
- Failure signals: tiny mixture weights, non-monotone log-likelihood, or clusters that change under nearby seeds
- Baseline: random assignments and a single spherical baseline

Practice prompts:
1. Change one hyperparameter by a small amount and explain whether the D5 score moves for a meaningful reason.

2. Add a random-label baseline to the results table and compare it with the method.

3. Pick one D5 point, print its assignment evidence, and decide whether the method is confident or ambiguous.